In [1]:
!git clone https://github.com/Anand-786/llm-quantization-thesis.git
%cd /content/llm-quantization-thesis
!git clone https://github.com/mit-han-lab/smoothquant.git smoothquant_repo
!pip uninstall smoothquant -y
!cd smoothquant_repo && pip install -e .
!pip install -q transformers accelerate datasets zstandard tqdm einops

from google.colab import drive
drive.mount('/content/drive')

import os, shutil
SAVE_DIR = "/content/drive/MyDrive/thesis_results/verification/falcon-7b"
os.makedirs(SAVE_DIR, exist_ok=True)

DRIVE_SCALES = "/content/drive/MyDrive/thesis_results/act_scales/falcon-7b.pt"
REPO_SCALES  = "/content/llm-quantization-thesis/smoothquant_repo/act_scales/falcon-7b.pt"
assert os.path.exists(DRIVE_SCALES), f"missing: {DRIVE_SCALES} — run generate_act_scales_cells.md first."
os.makedirs(os.path.dirname(REPO_SCALES), exist_ok=True)
shutil.copy2(DRIVE_SCALES, REPO_SCALES)

print("max scales :", REPO_SCALES)
!nvidia-smi

Cloning into 'llm-quantization-thesis'...
remote: Enumerating objects: 251, done.
remote: Counting objects: 100% (251/251), done.
remote: Compressing objects: 100% (165/165), done.
remote: Total 251 (delta 105), reused 211 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (251/251), 5.31 MiB | 28.29 MiB/s, done.
Resolving deltas: 100% (105/105), done.
/content/llm-quantization-thesis
Cloning into 'smoothquant_repo'...
remote: Enumerating objects: 352, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 352 (delta 120), reused 90 (delta 90), pack-reused 183 (from 1)
Receiving objects: 100% (352/352), 6.80 MiB | 30.27 MiB/s, done.
Resolving deltas: 100% (202/202), done.
Obtaining file:///content/llm-quantization-thesis/smoothquant_repo
  Preparing metadata (setup.py) ... done
  Running setup.py develop for smoothquant
Mounted at /content/drive
max scales : /content/llm-quantization-thesis/smoothquant_repo/act_scale

In [2]:
import re, torch, json

SCALES_PATH = "/content/llm-quantization-thesis/smoothquant_repo/act_scales/falcon-7b.pt"
SAVE_DIR    = "/content/drive/MyDrive/thesis_results/verification/falcon-7b"
ALPHA_MIN, ALPHA_MAX = 0.5, 0.9   # original range — paper α=0.6 sits inside this

raw = torch.load(SCALES_PATH, map_location="cpu")
LAYER_RE = re.compile(r"transformer\.h\.(\d+)\.(.+)")

sev_by_layer = {}
for name, vec in raw.items():
    m = LAYER_RE.match(name)
    if not m:
        continue
    suffix = m.group(2)
    if suffix not in ("self_attention.query_key_value", "mlp.dense_h_to_4h"):
        continue
    layer = int(m.group(1))
    v = vec.float().abs()
    s = (v.max() / v.median().clamp(min=1e-12)).item()
    sev_by_layer.setdefault(layer, []).append(s)

layers = sorted(sev_by_layer.keys())
sev = torch.tensor([sum(sev_by_layer[l]) / len(sev_by_layer[l]) for l in layers])

# Linear normalisation (same as Llama-2 final versions).
sev_norm = sev / sev.max()
alpha_per_layer = (ALPHA_MIN + (ALPHA_MAX - ALPHA_MIN) * sev_norm).tolist()

assert len(alpha_per_layer) == 32, f"expected 32 layers, got {len(alpha_per_layer)}"
print(f"per-layer α range: [{min(alpha_per_layer):.3f}, {max(alpha_per_layer):.3f}]  (32 layers)")
print(f"severity spread (max/min): {sev.max().item() / sev.min().item():.2f}×")
print()
print("layer  severity   α(l)")
for l, s, a in zip(layers, sev.tolist(), alpha_per_layer):
    print(f"  {l:2d}    {s:7.2f}    {a:.3f}")

with open(f"{SAVE_DIR}/falcon-7b_alpha_schedule.json", "w") as f:
    json.dump({
        "layers": layers,
        "severity": sev.tolist(),
        "alpha_per_layer": alpha_per_layer,
        "alpha_range": [ALPHA_MIN, ALPHA_MAX],
        "normalisation": "linear",
    }, f, indent=2)
print(f"\nschedule saved -> {SAVE_DIR}/falcon-7b_alpha_schedule.json")

per-layer α range: [0.634, 0.900]  (32 layers)
severity spread (max/min): 2.98×

layer  severity   α(l)
   0      14.81    0.900
   1      12.17    0.829
   2       7.83    0.711
   3       6.56    0.677
   4       7.54    0.704
   5       7.27    0.696
   6       8.47    0.729
   7       8.79    0.737
   8       9.66    0.761
   9       8.91    0.741
  10       8.36    0.726
  11       8.16    0.720
  12       8.36    0.726
  13       7.35    0.698
  14       7.58    0.705
  15       7.32    0.698
  16       8.06    0.718
  17       7.75    0.709
  18       7.31    0.697
  19       7.53    0.703
  20       7.01    0.689
  21       7.13    0.693
  22       7.25    0.696
  23       6.50    0.676
  24       6.39    0.673
  25       5.52    0.649
  26       4.98    0.634
  27       5.65    0.653
  28       5.41    0.646
  29       5.36    0.645
  30       5.05    0.636
  31       7.61    0.706

schedule saved -> /content/drive/MyDrive/thesis_results/verification/falcon-7b/falcon-7b_alpha_

In [3]:
%%writefile /content/smooth_per_layer_falcon.py
"""Per-layer α extension of smoothquant.smooth.smooth_lm (Falcon)."""
import re
import torch
from transformers.models.falcon.modeling_falcon import FalconDecoderLayer
from smoothquant.smooth import smooth_ln_fcs

LAYER_RE = re.compile(r"transformer\.h\.(\d+)$")

@torch.no_grad()
def smooth_lm_per_layer_falcon(model, scales, alpha_schedule):
    if not isinstance(alpha_schedule, (list, tuple)):
        raise TypeError("alpha_schedule must be a list/tuple of floats")
    for name, module in model.named_modules():
        if not isinstance(module, FalconDecoderLayer):
            continue
        m = LAYER_RE.match(name)
        if m is None:
            continue
        layer_idx = int(m.group(1))
        alpha = float(alpha_schedule[layer_idx])

        qkv = module.self_attention.query_key_value
        fc1 = module.mlp.dense_h_to_4h
        qkv_input_scales = scales[name + ".self_attention.query_key_value"]
        fc1_input_scales = scales[name + ".mlp.dense_h_to_4h"]

        if (not module.config.new_decoder_architecture
                and module.config.parallel_attn):
            # Falcon-7B path: one input_layernorm absorbs into both qkv and fc1.
            attn_ln = module.input_layernorm
            smooth_ln_fcs(attn_ln, [qkv, fc1], qkv_input_scales, alpha)
        else:
            # Falcon-40B-style path: separate norms for attn and ffn.
            attn_ln = (module.ln_attn if module.config.new_decoder_architecture
                       else module.input_layernorm)
            ffn_ln  = (module.ln_mlp  if module.config.new_decoder_architecture
                       else module.post_attention_layernorm)
            smooth_ln_fcs(attn_ln, qkv, qkv_input_scales, alpha)
            smooth_ln_fcs(ffn_ln,  fc1, fc1_input_scales, alpha)

Writing /content/smooth_per_layer_falcon.py


In [4]:
import sys
sys.path.insert(0, "/content")
sys.path.insert(0, "/content/llm-quantization-thesis/smoothquant_repo")

import torch, torch.nn as nn, json, time, tqdm, os
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from smoothquant.smooth import smooth_lm
from smoothquant.fake_quant import quantize_model
from smooth_per_layer_falcon import smooth_lm_per_layer_falcon

MODEL = "tiiuae/falcon-7b"
SCALES_PATH = "/content/llm-quantization-thesis/smoothquant_repo/act_scales/falcon-7b.pt"
SAVE_DIR    = "/content/drive/MyDrive/thesis_results/verification/falcon-7b"


class Evaluator:
    def __init__(self, dataset, tokenizer, device, n_samples=40):
        self.dataset = tokenizer("\n\n".join(dataset["text"]), return_tensors="pt").input_ids.to(device)
        self.n_samples = n_samples
    @torch.no_grad()
    def evaluate(self, model):
        model.eval()
        nlls = []
        n = self.n_samples
        for i in tqdm.tqdm(range(n), desc="PPL"):
            batch = self.dataset[:, (i * 2048):((i + 1) * 2048)].to(model.device)
            logits = model(batch).logits
            shift_logits = logits[:, :-1, :].contiguous().float()
            shift_labels = self.dataset[:, (i * 2048):((i + 1) * 2048)][:, 1:]
            loss = nn.CrossEntropyLoss()(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
            nlls.append(loss.float() * 2048)
        return torch.exp(torch.stack(nlls).sum() / (n * 2048))


print("Loading tokenizer + dataset + scales...")
tokenizer  = AutoTokenizer.from_pretrained(MODEL)

# Tokenizer sanity check — Falcon uses HF byte-BPE, no SentencePiece quirks expected.
_ids = tokenizer("The capital of France is", return_tensors="pt").input_ids[0].tolist()
_dec = tokenizer.decode(_ids)
print(f"tokenizer class: {type(tokenizer).__name__}")
print(f"sanity decode  : {_dec}")
assert "<unk>" not in _dec, "tokenizer is producing <unk> for spaces — bail out before wasting GPU time."

dataset    = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
evaluator  = Evaluator(dataset, tokenizer, "cuda")
act_scales = torch.load(SCALES_PATH)

C_QPARAMS   = dict(weight_quant="per_channel", act_quant="per_token", quantize_bmm_input=True)
PAPER_ALPHA = 0.60  # SmoothQuant paper Table 7 — Falcon-7B row

results = []  # appended to by each run cell below

def _run_config(label, smooth_spec, qparams):
    print(f"\n{'='*60}\n  {label}\n{'='*60}")
    t0 = time.time()
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="auto")

    if smooth_spec == "none":
        pass
    elif smooth_spec[0] == "max":
        smooth_lm(model, act_scales, smooth_spec[1])
    elif smooth_spec[0] == "perlayer":
        smooth_lm_per_layer_falcon(model, act_scales, smooth_spec[1])
    else:
        raise ValueError(f"unknown smooth spec: {smooth_spec}")

    if qparams is not None:
        model = quantize_model(model, **qparams)

    ppl = evaluator.evaluate(model).item()
    elapsed = time.time() - t0
    print(f">>> {label}: PPL = {ppl:.4f}  ({elapsed:.0f}s)")

    rec = {
        "model": MODEL,
        "label": label,
        "smooth": (smooth_spec if isinstance(smooth_spec, str)
                   else (smooth_spec[0] if smooth_spec[0] != "perlayer" else "perlayer")),
        "qparams": qparams,
        "ppl": round(ppl, 4),
        "seconds": round(elapsed, 1),
    }
    results.append(rec)
    with open(f"{SAVE_DIR}/falcon-7b_{label}.json", "w") as f:
        json.dump(rec, f, indent=2)

    del model
    torch.cuda.empty_cache()
    return rec

print("setup ready — run the three config cells below in order.")

Loading tokenizer + dataset + scales...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (329999 > 2048). Running this sequence through the model will result in indexing errors


setup ready — run the three config cells below in order.


In [5]:
_run_config("1_FP16", "none", None)

`torch_dtype` is deprecated! Use `dtype` instead!



  1_FP16


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

PPL: 100%|██████████| 40/40 [00:25<00:00,  1.54it/s]

>>> 1_FP16: PPL = 6.9476  (70s)


{'model': 'tiiuae/falcon-7b',
 'label': '1_FP16',
 'smooth': 'none',
 'qparams': None,
 'ppl': 6.9476,
 'seconds': 69.5}

In [6]:
_run_config(f"2_C_max_a{PAPER_ALPHA}", ("max", PAPER_ALPHA), C_QPARAMS)


  2_C_max_a0.6


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PPL: 100%|██████████| 40/40 [00:26<00:00,  1.49it/s]

>>> 2_C_max_a0.6: PPL = 6.9844  (291s)


{'model': 'tiiuae/falcon-7b',
 'label': '2_C_max_a0.6',
 'smooth': 'max',
 'qparams': {'weight_quant': 'per_channel',
  'act_quant': 'per_token',
  'quantize_bmm_input': True},
 'ppl': 6.9844,
 'seconds': 291.0}

In [8]:
_run_config("3_C_perlayer", ("perlayer", alpha_per_layer), C_QPARAMS)


  3_C_perlayer


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PPL: 100%|██████████| 40/40 [00:26<00:00,  1.49it/s]

>>> 3_C_perlayer: PPL = 6.9873  (290s)


{'model': 'tiiuae/falcon-7b',
 'label': '3_C_perlayer',
 'smooth': 'perlayer',
 'qparams': {'weight_quant': 'per_channel',
  'act_quant': 'per_token',
  'quantize_bmm_input': True},
 'ppl': 6.9873,
 'seconds': 290.2}

In [9]:
def ppl(label):
    return next(r for r in results if r["label"] == label)["ppl"]

PAPER_FP16, PAPER_SQ, PAPER_ALPHA = 6.590, 6.629, 0.60
our_fp16    = ppl("1_FP16")
our_paper   = ppl(f"2_C_max_a{PAPER_ALPHA}")
our_alpha_l = ppl("3_C_perlayer")

print(f"\n{'='*78}")
print(f"  Falcon-7B — WikiText-2 PPL — paper-comparable table")
print(f"{'='*78}")
print(f"\n{'config':<30} {'ours':>10} {'paper':>10} {'Δ ours−paper':>14}")
print("-" * 70)
print(f"{'FP16':<30} {our_fp16:>10.4f} {PAPER_FP16:>10.4f} {our_fp16 - PAPER_FP16:>+14.4f}")
print(f"{'C + max α=0.60 (paper cfg)':<30} {our_paper:>10.4f} {PAPER_SQ:>10.4f} {our_paper - PAPER_SQ:>+14.4f}")
print(f"{'C + per-layer α (ours)':<30} {our_alpha_l:>10.4f} {'—':>10} {'—':>14}")

print(f"\nDeltas vs OUR FP16 (within-session, noise-free):")
print(f"  C max α=0.60       − FP16  = {our_paper   - our_fp16:+.4f}")
print(f"  C per-layer α      − FP16  = {our_alpha_l - our_fp16:+.4f}")
print(f"  C per-layer        − C max = {our_alpha_l - our_paper:+.4f}  (negative → α(l) wins)")

print(f"\nPaper-gap reference:")
print(f"  paper W8A8 − paper FP16    = {PAPER_SQ - PAPER_FP16:+.4f}  (Table 7)")

shift = our_fp16 - PAPER_FP16
print(f"\nProtocol-shift diagnostic:")
print(f"  our FP16 − paper FP16  = {shift:+.4f}")
if abs(shift) < 0.05:
    print(f"  → protocols match; can cite paper numbers directly.")
else:
    print(f"  → protocols differ by ~{shift:+.3f} PPL; compare *within-session* deltas only.")

with open(f"{SAVE_DIR}/falcon-7b_summary.json", "w") as f:
    json.dump({"results": results, "alpha_per_layer": alpha_per_layer}, f, indent=2)
print(f"\nsaved -> {SAVE_DIR}/falcon-7b_summary.json")


  Falcon-7B — WikiText-2 PPL — paper-comparable table

config                               ours      paper   Δ ours−paper
----------------------------------------------------------------------
FP16                               6.9476     6.5900        +0.3576
C + max α=0.60 (paper cfg)         6.9844     6.6290        +0.3554
C + per-layer α (ours)             6.9873          —              —

Deltas vs OUR FP16 (within-session, noise-free):
  C max α=0.60       − FP16  = +0.0368
  C per-layer α      − FP16  = +0.0397
  C per-layer        − C max = +0.0029  (negative → α(l) wins)

Paper-gap reference:
  paper W8A8 − paper FP16    = +0.0390  (Table 7)

Protocol-shift diagnostic:
  our FP16 − paper FP16  = +0.3576
  → protocols differ by ~+0.358 PPL; compare *within-session* deltas only.

saved -> /content/drive/MyDrive/thesis_results/verification/falcon-7b/falcon-7b_summary.json
